In [91]:
print(9*9)

81


In [92]:

# from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.embeddings import FakeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [93]:
import os
# GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
INDEX_NAME = "ak-personal-1" # Ensure this index exists in your Pinecone console


In [94]:
## delete

# INDEX_NAME = "ak-personal-1"
# pc = Pinecone(api_key=PINECONE_API_KEY)
# index = pc.Index(INDEX_NAME)
# index.delete(delete_all=True)


In [95]:
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="models/embedding-001", 
#     google_api_key=GOOGLE_API_KEY
# )

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") 

In [96]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists, if not, create it
if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    print(f"Index '{INDEX_NAME}' not found. Creating it now...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=384, # MUST match the embedding model (384 for MiniLM, 768 for Google)
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1" # Starter plan usually requires us-east-1
        )
    )
    # Wait until the index is fully initialized and ready to receive data
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print("Waiting for Pinecone index to initialize...")
        time.sleep(2)
    print("Index created successfully!")

# Now initialize the VectorStore safely

# embeddings = FakeEmbeddings(size=768)
# vectorstore = InMemoryVectorStore(embeddings)
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    pinecone_api_key=PINECONE_API_KEY
)

In [97]:
# get text from PDF and vectorize them

file_path = r"D:\Code\ak\langchain-academy\my_data\nwpde_svm.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

# chunk the docs
text_splittter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
my_docs = text_splittter.split_documents(pages)

print(f"we split into {len(my_docs)} chunks")
vectorstore.add_documents(my_docs)

we split into 73 chunks


['e8115266-6027-46c2-b8df-28fa0bd0ff81',
 'd50ae61d-9f1f-4ac9-a133-758fe9c96bf0',
 'd6f3b223-a701-4341-b3ec-d0a9b6b10a83',
 '67bdaf97-637e-4e02-9d3e-c118c79093bf',
 '619b1ddb-5782-4510-b75b-653398ffea0c',
 '0d8bfbf6-148e-4b4f-8a6c-ca63ea9aaa6a',
 'a9c8804b-9b66-4654-9c29-097139b83991',
 '98d1ea66-21ab-4160-a755-2135c6edcbe1',
 '73c33fa2-9e9a-45eb-9690-0cea5ed956e0',
 'a36c5879-8aca-40a4-9207-478e07494b37',
 'd5c410ae-7ade-4498-bdb1-42373cfae0b2',
 'c9f02bd8-7956-452c-8293-2dd92e60f3ca',
 '82193f55-255e-453d-8827-83c259af1936',
 '9a5bef3e-55f7-4447-a7b4-bfc7897d7013',
 'd5fc800d-f3e8-40c7-82e7-63d91a256f31',
 'ced3694a-8df1-4b00-bd0b-d3f420696542',
 'e4c3206f-75b4-4a51-b65b-7e480072882d',
 '0f74cfc8-fa8c-4837-9b5e-9af0a092d023',
 '4bb99d01-8241-469e-87a5-816da16fd165',
 '9c4269f7-ca37-49a2-a26d-ce222a1396d4',
 '852cf7a8-718e-443c-a604-5f5aef176ed9',
 '8e9b9296-ba0e-420d-a28a-edc07eac4773',
 '41af429a-e83f-410f-a354-6133f2645385',
 'd689e3a2-152b-4a39-8f5b-75ff4f8aa1fa',
 '19e3ba8b-b8a1-

In [98]:
# 2. DEFINE THE TOOL
@tool
def lookup_policy(query: str, k:int = 3):
    """Consult the internal doc PDF to get context on numbers."""
    docs = vectorstore.similarity_search(query, k=k)
    
    if not docs:
        print("no chunks retrieved")
    else:
        all_chunks = "\n\n --- \n\n".join([doc.page_content for doc in docs])
    
    return all_chunks

tools = [lookup_policy]

llm_with_tools = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
).bind_tools(tools)

In [99]:
# query = "Explain the essence behind the SVM idea ?"
query = "According to the document, what is the full form of SVM??"
# query = "How to meaningfully add the supplementary variable in SVM?"

messages = [HumanMessage(content=query )]
ai_response = llm_with_tools.invoke(messages)
messages.append(ai_response)
print(ai_response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 1.477211098s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo

In [ ]:
tool_response = ai_response.tool_calls
tool_response

[{'name': 'lookup_policy',
  'args': {'query': 'supplementary variable in SVM'},
  'id': 'd5810688-1057-4773-aacb-5e79cacf943a',
  'type': 'tool_call'}]

In [ ]:
#loop thru all tool calls
for tool_call in tool_response:

    if tool_call:
        tool_args = tool_call["args"]
        tool_query = tool_call["args"]["query"]
        tool_id = tool_call["id"]

        tool_output = lookup_policy.invoke(tool_args)
        
        tool_msg = ToolMessage(content=tool_output, tool_call_id=tool_id)
        messages.append(tool_msg)


In [ ]:
# final tool-info powered llm call
response = llm_with_tools.invoke(messages)
print(response.content)

In [ ]:
tool_msg

ToolMessage(content='rate. Solving these overall set of equations in a systematic manner results in a solution that well\nrespects the energy structure of the physical system. However, in thlivis process we end up making\nthe system over-determined. Thus we add certain perturbation variables called supplementary\nvariables to make the system well-determined. This method is called Supplementary Variable\nMethod (SVM).\n1.1 Illustrating the SVM method\n\n --- \n\nrate. Solving these overall set of equations in a systematic manner results in a solution that well\nrespects the energy structure of the physical system. However, in thlivis process we end up making\nthe system over-determined. Thus we add certain perturbation variables called supplementary\nvariables to make the system well-determined. This method is called Supplementary Variable\nMethod (SVM).\n1.1 Illustrating the SVM method\n\n --- \n\n2. SVM correction:\nδ+\nt Φn = −M\n\x12\nLΦn+ 1\n2 + δf\nδΦ\n\x14\nΦ\nn+ 1\n2\n∗\n\x15\x1